# Metric Views & Semantic Layer Workshop

## What are Metric Views?

Metric Views are Unity Catalog objects that centralize business metric definitions in a **Semantic Layer**. Unlike standard views that lock in aggregations at creation time, metric views separate:

* **Measures** (aggregate calculations like SUM, COUNT, AVG)
* **Dimensions** (categorical attributes for grouping)

This separation enables flexible analysis at query time while ensuring consistent metric definitions across all downstream assets (dashboards, notebooks, Genie spaces).

## Key Benefits

✅ **Consistency**: Define metrics once, use everywhere  
✅ **Flexibility**: Query any combination of dimensions and measures  
✅ **Governance**: Centralized metric definitions with Unity Catalog permissions  
✅ **Performance**: Optional materialization for faster queries  
✅ **AI-Ready**: Semantic metadata (synonyms, display names) for Genie and AI tools

---

## Workshop Dataset: Coffee Shop Analytics

In this workshop, we'll explore the `catalog.schema.coffee_metric_view` which tracks:

* **Transactions** from multiple coffee shop stores
* **Products** sold (with details)
* **Store information** and reviews
* **Time dimensions** (date, hour of day)

### Metric View Structure

**Dimensions:**
* `product` - Product ID
* `Product detail` - Product description
* `store_name` - Store name
* `month_day` - Transaction date
* `hour_of_day` - Hour when transaction occurred

**Measures:**
* `total_transactions` - Count of transactions
* `total_sales` - Total revenue
* `total_quantity_sold` - Total items sold

## Section 1: Basic Querying Syntax

### Critical Rules for Querying Metric Views

1. **Wrap measures with MEASURE()** - `MEASURE(\`total_sales\`)`
2. **Always use GROUP BY ALL** - Required for all queries
3. **Never use SELECT \*** - Explicitly list dimensions and measures
4. **Backtick all names** - Especially names with spaces
5. **No nested aggregations** - MIN/MAX/AVG on measures not allowed

Let's start with simple queries and build up complexity!

In [0]:
%sql
USE CATALOG dbacademy; -- Need to be updated 
Use SCHEMA <insert_your_schema>; -- Need to be updated

In [0]:
-- Basic aggregation: Sales performance by store
-- This demonstrates grouping by a dimension and selecting multiple measures
SELECT 
  `store_name`,
  MEASURE(`total_sales`) AS revenue,
  MEASURE(`total_transactions`) AS transactions,
  MEASURE(`total_quantity_sold`) AS items_sold
FROM coffee_metric_view
GROUP BY ALL
ORDER BY revenue DESC

store_name,revenue,transactions,items_sold
Inferno Brews,236511.17,3775279008,71737.0
Astoria Brews,232243.91,3797253689,70991.0
Lower Grounds Coffee Co,230092.89,3572005247,71742.0


In [0]:
-- Multiple dimensions: Product performance with details
SELECT 
  `product`,
  `Product detail`,
  MEASURE(`total_sales`) AS revenue,
  MEASURE(`total_quantity_sold`) AS quantity
FROM coffee_metric_view
GROUP BY ALL
ORDER BY revenue DESC
LIMIT 10

product,Product detail,revenue,quantity
61,Latte,21151.75,4453.0
59,Jumbo Savory Scone,21006.0,4668.0
39,Ethiopia Sm,19112.25,4497.0
41,Ginger Biscotti,17641.75,4151.0
55,Jamacian Coffee River,17384.0,4346.0
38,Ethiopia Rg,17257.5,4602.0
36,Ethiopia,16481.25,4395.0
60,Jumbo Savory Scone,16233.75,4329.0
40,Ginger Biscotti,15997.5,4266.0
27,Dark chocolate Rg,15109.5,4317.0


## Section 2: Time-Based Analysis

Metric views excel at time-series analysis. Let's explore sales patterns over time and by hour of day.

In [0]:
-- Time-based analysis: Peak hours analysis
-- Shows when customers buy the most throughout the day
SELECT 
  `hour_of_day`,
  MEASURE(`total_sales`) AS revenue,
  MEASURE(`total_transactions`) AS transactions,
  ROUND(MEASURE(`total_sales`) / MEASURE(`total_transactions`), 2) AS avg_transaction_value
FROM coffee_metric_view
GROUP BY ALL
ORDER BY `hour_of_day`

hour_of_day,revenue,transactions,avg_transaction_value
6,21900.27,353149265,0.0
7,63531.33,1012562161,0.0
8,82702.03,1324752516,0.0
9,85177.09,1335028367,0.0
10,88681.49,1405563740,0.0
11,46322.38,723920337,0.0
12,40192.79,636612962,0.0
13,40367.45,650696177,0.0
14,41304.74,661770685,0.0
15,41736.34,665801583,0.0


## Section 3: Advanced Query Patterns

Let's explore sophisticated analytical patterns:

* **Calculated metrics** combining multiple measures
* **Top-K queries** using window functions for ranking

In [0]:
-- Calculate average transaction value and price per item
SELECT 
  `store_name`,
  MEASURE(`total_sales`) AS total_revenue,
  MEASURE(`total_transactions`) AS transactions,
  MEASURE(`total_quantity_sold`) AS items_sold,
  ROUND(MEASURE(`total_sales`) / MEASURE(`total_transactions`), 2) AS avg_transaction_value,
  ROUND(MEASURE(`total_sales`) / MEASURE(`total_quantity_sold`), 2) AS avg_price_per_item
FROM coffee_metric_view
GROUP BY ALL
ORDER BY total_revenue DESC

store_name,total_revenue,transactions,items_sold,avg_transaction_value,avg_price_per_item
Inferno Brews,236511.17,3775279008,71737.0,0.0,3.3
Astoria Brews,232243.91,3797253689,70991.0,0.0,3.27
Lower Grounds Coffee Co,230092.89,3572005247,71742.0,0.0,3.21


In [0]:
-- Top-K query: Best selling products using ROW_NUMBER()
WITH ranked_products AS (
  SELECT 
    `Product detail`,
    MEASURE(`total_sales`) AS revenue,
    MEASURE(`total_quantity_sold`) AS quantity,
    ROW_NUMBER() OVER (ORDER BY MEASURE(`total_sales`) DESC) AS rank
  FROM coffee_metric_view
  GROUP BY ALL
)
SELECT 
  rank,
  `Product detail`,
  revenue,
  quantity
FROM ranked_products
WHERE rank <= 5

rank,Product detail,revenue,quantity
1,Jumbo Savory Scone,37239.75,8997.0
2,Ginger Scone,34489.5,12978.0
3,Organic Decaf Blend,33648.46,9639.0
4,Ginger Biscotti,33639.25,8417.0
5,I Need My Bean! Diner mug,24697.0,9017.0


## Section 4: Creating Metric Views

### YAML Structure

Metric views are defined using SQL DDL with YAML:

```sql
CREATE OR REPLACE VIEW catalog.schema.metric_view_name
WITH METRICS
LANGUAGE YAML
AS $$
  version: 1.1
  source: catalog.schema.source_table
  dimensions:
    - name: dimension_name
      expr: column_or_expression
      display_name: Human Readable Name  # Optional
  measures:
    - name: measure_name
      expr: aggregate_expression
      format:                            # Optional
        type: currency
        currency_code: USD
$$
```

**Key Fields:**
* `version: 1.1` - Latest version
* `source` - Base table or SQL query
* `dimensions` - Categorical attributes
* `measures` - Aggregate calculations
* `joins` - Star/snowflake schema (optional)
* `display_name`, `comment`, `synonyms` - Semantic metadata (optional)

### Our Coffee Shop Metric View

The `users.tayyab_ali.coffee_metric_view` demonstrates:

**Star Schema with Joins:**
* Fact table: `fct_coffee_shop_transactions`
* Dimension tables: `dim_coffee_shop_stores`, `dim_coffee_products`
* Nested join: Store reviews (snowflake pattern)

**Dimensions:**
* `product`, `Product detail` - Product information
* `store_name` - Store name from dimension table
* `month_day` - Derived from `transaction_date`
* `hour_of_day` - Extracted from `transaction_time`

**Measures:**
* `total_transactions` - SUM(transaction_id)
* `total_sales` - ROUND(sum(transation_total), 2)
* `total_quantity_sold` - SUM(transaction_qty)

## Section 5: Semantic Layer & Best Practices

### What is a Semantic Layer?

A **Semantic Layer** translates technical schemas into business terminology, providing:
* Consistent metric definitions across the organization
* Self-service analytics for non-technical users
* AI-ready metadata for tools like Genie

### Semantic Metadata

Enhance metric views with:

**Display Names & Comments:**
```yaml
measures:
  - name: total_sales
    expr: SUM(amount)
    display_name: Total Revenue
    comment: Sum of all transaction amounts in USD
```

**Formatting:**
```yaml
format:
  type: currency
  currency_code: USD
  decimal_places:
    type: exact
    places: 2
```

**Synonyms** (for AI/Genie):
```yaml
synonyms:
  - revenue
  - sales
```

### Best Practices

1. **Use clear, business-friendly names** - Avoid technical jargon
2. **Document everything** - Add comments to dimensions and measures
3. **Compose metrics** - Reference other measures using MEASURE()
4. **Use materialization** - Pre-compute for expensive queries
5. **Control access** - Use Unity Catalog permissions
6. **Integrate everywhere** - Use in Lakeview Dashboards, Genie, SQL, Python

## Section 6: Hands-On Exercises

Try these exercises to practice querying metric views:

### Exercise 1: Peak Hours Analysis
**Goal**: Find which hour of the day has the highest average transaction value

**Hint**: Calculate `total_sales / total_transactions` grouped by `hour_of_day`

### Exercise 2: Product Performance Matrix
**Goal**: Create a report showing each product's revenue, quantity sold, and average price

**Hint**: Use all three measures and calculate `total_sales / total_quantity_sold`

---

**Try writing these queries in the cells below!**

In [0]:
-- Exercise 1: Peak Hours Analysis
-- Write your query here



In [0]:
-- Exercise 2: Product Performance Matrix
-- Write your query here



## Workshop Summary

### Key Takeaways

✅ **Metric Views** separate measures from dimensions for flexible analysis  
✅ **Query Syntax**: MEASURE() wrapper, GROUP BY ALL, backticks required  
✅ **Advanced Patterns**: Calculated metrics, Top-K with ROW_NUMBER()  
✅ **YAML Definition**: version, source, dimensions, measures, joins  
✅ **Semantic Layer**: Business-friendly metadata powers Genie and dashboards

### Next Steps

1. Run the queries in this notebook
2. Create metric views for your datasets
3. Build Lakeview dashboards
4. Try Genie with natural language queries
5. Add materialization for performance

### Resources

* [Metric Views Documentation](https://docs.databricks.com/en/sql/language-manual/sql-ref-metric-views.html)
* [Unity Catalog Best Practices](https://docs.databricks.com/en/data-governance/unity-catalog/best-practices.html)
* [Genie Spaces](https://docs.databricks.com/en/genie/index.html)

**Happy Analyzing! ☕️📊**

In [0]:
-- Solution 1: Peak Hours Analysis
SELECT 
  `hour_of_day`,
  MEASURE(`total_sales`) AS total_revenue,
  MEASURE(`total_transactions`) AS transactions,
  ROUND(MEASURE(`total_sales`) / MEASURE(`total_transactions`), 2) AS avg_transaction_value
FROM coffee_metric_view
GROUP BY ALL
ORDER BY avg_transaction_value DESC

hour_of_day,total_revenue,transactions,avg_transaction_value
16,41122.75,673466024,0.0
6,21900.27,353149265,0.0
13,40367.45,650696177,0.0
15,41736.34,665801583,0.0
17,40134.31,644138922,0.0
19,28450.46,452082196,0.0
7,63531.33,1012562161,0.0
10,88681.49,1405563740,0.0
9,85177.09,1335028367,0.0
14,41304.74,661770685,0.0


In [0]:
-- Solution 2: Product Performance Matrix
SELECT 
  `Product detail`,
  MEASURE(`total_sales`) AS revenue,
  MEASURE(`total_quantity_sold`) AS quantity_sold,
  ROUND(MEASURE(`total_sales`) / MEASURE(`total_quantity_sold`), 2) AS avg_price_per_item
FROM coffee_metric_view
GROUP BY ALL
ORDER BY revenue DESC

Product detail,revenue,quantity_sold,avg_price_per_item
Jumbo Savory Scone,37239.75,8997.0,4.14
Ginger Scone,34489.5,12978.0,2.66
Organic Decaf Blend,33648.46,9639.0,3.49
Ginger Biscotti,33639.25,8417.0,4.0
I Need My Bean! Diner mug,24697.0,9017.0,2.74
I Need My Bean! T-shirt,24129.5,8817.0,2.74
I Need My Bean! Latte cup,24015.0,8757.0,2.74
Hazelnut Biscotti,23852.5,8697.0,2.74
Croissant,23016.0,8407.0,2.74
Latte,21151.75,4453.0,4.75
